In [1]:
import nn_fncs
mat_data = nn_fncs.read_mat_workspace('Thrust_data.mat')
nn_in = mat_data.get('nn_in')
nn_out = mat_data.get('nn_out')
# Get every 5th data sample
nn_in = nn_in[:, ::5, :]
nn_out = nn_out[:, ::5, :]

print(f'nn_in shape: {nn_in.shape}')  # (num_trajectories, num_time_steps, num_inputs)
print(f'nn_out shape: {nn_out.shape}')      # (num_trajectories, num_time_steps, num_outputs)


nn_in shape: (329, 2000, 16)
nn_out shape: (329, 2000, 12)


In [2]:
# MODEL STRUCTURE

import torch
import torch.nn as nn
import torch.nn.functional as F

n_neurons = 32
class ThrustModel(nn.Module):
    def __init__(self, in_size=1, out_size=3):  
        super(ThrustModel, self).__init__()
        # Fully connected layers
        self.fc1 = nn.Linear(in_size, n_neurons) 
        self.fc2 = nn.Linear(n_neurons, n_neurons)
        self.fc3 = nn.Linear(n_neurons, n_neurons)
        self.fc4 = nn.Linear(n_neurons, n_neurons)
        self.fc5 = nn.Linear(n_neurons, n_neurons)
        self.fc6 = nn.Linear(n_neurons, n_neurons)
        self.fc7 = nn.Linear(n_neurons, n_neurons)
        self.fc8 = nn.Linear(n_neurons, n_neurons)
        self.fc9 = nn.Linear(n_neurons, n_neurons)
        self.output = nn.Linear(n_neurons, out_size)
        self.dropout = nn.Dropout(0.1)
        
    def forward(self, x):
        
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        x = self.dropout(x)
        x = F.relu(self.fc3(x))
        x = self.dropout(x)
        x = F.relu(self.fc3(x))
        x = self.dropout(x)
        x = F.relu(self.fc4(x))
        x = self.dropout(x)
        x = F.relu(self.fc5(x))
        x = self.dropout(x)
        x = F.relu(self.fc6(x))
        x = self.dropout(x)
        x = F.relu(self.fc7(x))
        x = self.dropout(x)
        x = F.relu(self.fc8(x))
        x = self.dropout(x)
        x = F.relu(self.fc9(x))
        x = self.dropout(x)
        x = self.output(x)
        return x

model = ThrustModel(nn_in.shape[2], nn_out.shape[2])

# Print model summary
print(model)

# Count trainable parameters
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total trainable parameters: {total_params}')

ThrustModel(
  (fc1): Linear(in_features=16, out_features=32, bias=True)
  (fc2): Linear(in_features=32, out_features=32, bias=True)
  (fc3): Linear(in_features=32, out_features=32, bias=True)
  (fc4): Linear(in_features=32, out_features=32, bias=True)
  (fc5): Linear(in_features=32, out_features=32, bias=True)
  (fc6): Linear(in_features=32, out_features=32, bias=True)
  (fc7): Linear(in_features=32, out_features=32, bias=True)
  (fc8): Linear(in_features=32, out_features=32, bias=True)
  (fc9): Linear(in_features=32, out_features=32, bias=True)
  (output): Linear(in_features=32, out_features=12, bias=True)
  (dropout): Dropout(p=0.1, inplace=False)
)
Total trainable parameters: 9388


In [3]:
# One step training
import torch.optim as optim
from torcheval.metrics.functional import mean_squared_error, r2_score
from scipy.integrate import odeint

def one_step_training(model, criterion, optimizer, input_data, target):
    t = 0
    loss = 0.0
    xk = target[0]
    predictions = torch.zeros_like(target)
    for i in range(target.shape[0]-1):

        with torch.set_grad_enabled(True):
            # Forward pass
            pred = model(input_data)  # Integrate over a small time step
            loss = criterion(pred, target)
            # Backward and optimize
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
    r2 = r2_score(pred, target)
    return pred, loss, r2

In [4]:
# Example training step
from random import randint
id = randint(0, nn_in.shape[0]-1)
print(f'Training on trajectory id: {id}')
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
pred, loss, r2 = one_step_training(model, 
                                   criterion, 
                                   optimizer, 
                                   torch.tensor(nn_in[id], dtype=torch.float32), 
                                   torch.tensor(nn_out[id], dtype=torch.float32))

print(f'Loss: {loss.item()}, R2: {r2.item()}')

Training on trajectory id: 24
Loss: 0.06200530007481575, R2: -inf


In [ ]:
# Complete training loop
num_epochs = 100

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
model.train()
nn_in_tensor = torch.from_numpy(nn_in).type(torch.float32).to(device)
nn_out_tensor = torch.from_numpy(nn_out).type(torch.float32).to(device)
history_loss = []
for epoch in range(num_epochs):
    epoch_loss = 0.0
    epoch_r2 = 0.0
    # for traj in range(nn_in_tensor.shape[0]):
    for i in range(20):
        id = randint(0, nn_in_tensor.shape[0]-1)
        pred, loss, r2 = one_step_training(model, 
                                            criterion, 
                                            optimizer,
                                            nn_in_tensor[id], 
                                            nn_out_tensor[id])
        epoch_loss += loss.item()
        epoch_r2 += r2.item()
    epoch_loss /= (nn_in_tensor.shape[0])
    epoch_r2 /= (nn_in_tensor.shape[0])
    #if (epoch+1) % 100 == 0:
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.6f}, R2: {epoch_r2:.6f}')
    history_loss.append(epoch_loss)


Epoch [1/100], Loss: 0.000586, R2: nan
Epoch [2/100], Loss: 0.001889, R2: -inf
Epoch [3/100], Loss: 0.001679, R2: -inf
Epoch [4/100], Loss: 0.001959, R2: -inf
Epoch [5/100], Loss: 0.001195, R2: -inf
Epoch [6/100], Loss: 0.002333, R2: nan
Epoch [7/100], Loss: 0.001572, R2: nan
Epoch [8/100], Loss: 0.001806, R2: -inf
Epoch [9/100], Loss: 0.001860, R2: -inf
Epoch [10/100], Loss: 0.001187, R2: -inf
Epoch [11/100], Loss: 0.001397, R2: -inf
Epoch [12/100], Loss: 0.001602, R2: -inf
Epoch [13/100], Loss: 0.001663, R2: nan
Epoch [14/100], Loss: 0.001093, R2: -inf
Epoch [15/100], Loss: 0.001463, R2: -inf
Epoch [16/100], Loss: 0.003823, R2: -inf
Epoch [17/100], Loss: 0.007930, R2: -inf
Epoch [18/100], Loss: 0.006751, R2: -inf
Epoch [19/100], Loss: 0.003224, R2: -inf
Epoch [20/100], Loss: 0.003760, R2: -inf
Epoch [21/100], Loss: 0.002386, R2: -inf
Epoch [22/100], Loss: 0.006373, R2: -inf
Epoch [23/100], Loss: 0.007538, R2: -inf
Epoch [24/100], Loss: 0.011739, R2: -inf
Epoch [25/100], Loss: 0.01485

In [ ]:

# plot training loss
import matplotlib.pyplot as plt
plt.plot(history_loss)